# Data Quality Assessment

We'll start by quantifying the impact of missing values and performing a frequency analysis on negative quantities to confirm the "cancellation" hypothesis. This will help us understand the extent of data quality issues and guide our data cleaning and preparation efforts.

In [228]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("../../data/raw/online_retail_II.csv" , encoding="ISO-8859-1")

We're going to convert the invoice date column to a datetime format and check for any missing values in the dataset. This will allow us to identify any potential issues with the data and ensure that we have a complete and accurate dataset for our analysis.

In [229]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

In [230]:
df.isna().sum() / len(df) * 100

Invoice         0.000000
StockCode       0.000000
Description     0.268310
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    24.926648
Country         0.000000
dtype: float64

We can analyze the missing values in the dataset are minimal, with only a high percentage in Customer ID and lower percentages in Description. Therefore, we can consider replace Description with a placeholder value like "Unknown" or "No Description" to maintain the integrity of the dataset. For Customer ID, we can consider dropping those rows or imputing them based on other available information.

Meanwhile, we have to investigate why are there 24.92% in Customer ID. This might be related to the nature of the dataset, where some transactions may not have associated customer information. We can explore this further by analyzing the distribution of Customer ID and checking for any patterns or correlations with other variables.

In [231]:
df[df['Customer ID'].isna()].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom


### What do we do with the Customer ID missing values?

The purpose of this analysis is to understand the customer segments and their purchasing behavior. Since Customer ID is a crucial identifier for segmenting customers, we need to carefully consider how to handle the missing values.

So, we're going to keep the rows with missing Customer ID for now, but when we perform the customer segmentation analysis, we will exclude those rows from the analysis. This way, we can still analyze the transactions and their characteristics without losing valuable information.

### What do we do with the Description missing values?

While the missing values in Description are relatively low, we can replace them with a placeholder value like "Unknown" or "No Description" to maintain the integrity of the dataset. This will allow us to retain those rows for analysis without losing any valuable information.

In [232]:
print(f"Number of duplicated rows: {df.duplicated().sum()}")
df.apply(lambda col: col.value_counts().gt(1).sum())

Number of duplicated rows: 5268


Invoice        20059
StockCode       3837
Description     3915
Quantity         414
InvoiceDate    19018
Price            526
Customer ID     4293
Country           38
dtype: int64

In [233]:
duplicate_rows = df[
    df.duplicated(
        subset=[
            'Invoice',
            'StockCode',
            'Description',
            'Price',
            'Customer ID',
            'Quantity',
        ],
        keep=False,
    )
]

print(f"Number of duplicated rows: {len(duplicate_rows)}")
print(f"Percentage of duplicated rows: {len(duplicate_rows) / len(df) * 100:.2f}%")

Number of duplicated rows: 10149
Percentage of duplicated rows: 1.87%


We have 10149 duplicated rows in the dataset, which is only a 1.87% of the total dataset. We can consider dropping those rows to maintain the integrity of the dataset and avoid any potential bias in our analysis. By removing duplicates, we can ensure that our analysis is based on unique transactions and customer interactions, leading to more accurate insights into customer segments and purchasing behavior.

In [234]:
df = df.drop_duplicates(
    subset=[
                'Invoice',
                'StockCode',
                'Description',
                'Price',
                'Customer ID',
                'Quantity',
            ],
    keep='first'
)

print(f"Number of duplicated rows after dropping duplicates: {df.duplicated().sum()}")
print(f"Percentage of duplicated rows after dropping duplicates: {df.duplicated().sum() / len(df) * 100:.2f}%")

Number of duplicated rows after dropping duplicates: 0
Percentage of duplicated rows after dropping duplicates: 0.00%


We can also check that the duplicated values are part of the business logic, as they are related to the same invoice number. Therefore, we will not remove them from the dataset.

In [235]:
invalid_dates = df[
    (df["InvoiceDate"] < "2009-12-01") &
    (df["InvoiceDate"] > "2011-12-10")
]

print(f"Number of invalid dates: {len(invalid_dates)}")

Number of invalid dates: 0


There's no invalid dates in the dataset, as all the dates are within the expected range **(between 2009-12-01 and 2011-12-10)**. This indicates that the date information is reliable and can be used for further analysis without any concerns about data quality issues related to dates.

In [236]:
df[df["Quantity"] <= 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


Now, we can analyze why the quantity is lower than 0, which might be related to the nature of the dataset, because there's a code in the invoice number that starts with "C", which indicates a cancellation. Therefore, we can consider those rows as valid transactions and keep them in the dataset for analysis, but we're going to create later a new column to indicate whether the transaction is a cancellation or not, based on the invoice number. This will allow us to analyze the cancellations separately and understand their impact on the overall customer segments and purchasing behavior.

Also, we will check if the negative quantities are related to the invoice numbers that start with "C", which indicates a cancellation. If we find **some negative quantities that are not related to cancellations**, we will investigate further to understand the reason behind those transactions and determine if they should be included in the analysis or not.

On other hand, we discover a new type of StockCode called "D", which is related to discounts, and give us a quantity in negative, so this indicate that the negative quantities are not only related to cancellations, but also to discounts. Therefore, we will need to consider both cancellations and discounts when analyzing the negative quantities in the dataset.

In [237]:
percent_cancellations = df[df["Quantity"] <= 0]["Invoice"].str.startswith("C").sum() / len(df[df["Quantity"] < 0]) * 100

print(f"Percentage of cancellations among negative quantities: {percent_cancellations:.2f}%")

Percentage of cancellations among negative quantities: 87.38%


The percentage of cancellations among negative quantities with invoice numbers starting with "C" is **87.38%**, which indicates that the majority of negative quantities are indeed related to cancellations. However, there are still some negative quantities that are not associated with cancellations, which we're going to investigate further to understand their nature

In [238]:
not_invoice_cancellations = df[(df["Quantity"] <= 0) & (~df["Invoice"].str.startswith("C"))]
print(f"Number of negative quantities not related to invoice cancellations: {len(not_invoice_cancellations)}")
print(f"Percentage of negative quantities not related with the total of invoice cancellations: {len(not_invoice_cancellations) / len(df[df['Quantity'] < 0]) * 100:.2f}%")

Number of negative quantities not related to invoice cancellations: 1336
Percentage of negative quantities not related with the total of invoice cancellations: 12.62%


In summary, it's only the 12.62% of the negative quantities that are not related to invoice cancellations, which is a relatively small percentage. Therefore, we can drop those rows from the dataset, as they may not be relevant to our analysis and could introduce noise or bias into our results. By focusing on the valid transactions and cancellations, we can gain a clearer understanding of customer segments and their purchasing behavior.

In [239]:
negative_prices = df[df["Price"] <= 0].head()
print(f"Number of negative prices: {len(negative_prices)}")
print(f"Percentage of negative prices: {len(negative_prices) / len(df) * 100:.5f}%")
negative_prices.head()

Number of negative prices: 5
Percentage of negative prices: 0.00093%


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1510,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1985,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1986,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
2022,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


We can also analyze the negative prices in the dataset, which we can look into further and we only have **0.00093%** of the dataset. We conclude that the negative prices are likely to be data entry errors or anomalies, and we can consider dropping those rows because they are not related with some special stock codes.Therefore, we should remove these rows from the dataset to maintain the integrity of the analysis. By removing these outliers, we can ensure that our analysis is based on accurate and reliable data, leading to more meaningful insights into customer segments and purchasing behavior.

In [240]:
df[(df["StockCode"].str.len() <= 4)]["StockCode"].value_counts()

StockCode
POST    1257
DOT      710
M        566
C2       144
D         77
S         62
CRUK      16
PADS       4
B          3
m          1
Name: count, dtype: int64

We found that the company uses a small number of StockCodes with less than 5 characters, which are likely to be related to discounts or other special cases.

## Types

- **POST** : Postage
- **DOT** : Dotcom Postage
- **M** : Manual 
- **C2** : Carriage
- **D** : Discount
- **S** : Samples
- **CRUK** : CRUK Commission
- **PADS** : PADS TO MATCH ALL CUSHIONS
- **B** : Adjust bad debt

Therefore, we can conclude that there are many types of stock codes with give us more context about the nature of the transactions and their impact on the overall customer segments and purchasing behavior. By analyzing these stock codes, we can gain a better understanding of the different types of transactions and their significance in the dataset.

In [241]:
special_stock_codes = ["POST", "DOT", "M", "C2", "D", "S", "CRUK", "PADS", "B", "m"]

df[df["StockCode"].isin(
    special_stock_codes
)][
    ["StockCode", "Description"]
].drop_duplicates().sort_values("StockCode")

,StockCode,Description
299982,B,Adjust bad debt
1423,C2,CARRIAGE
453999,C2,NaN
317508,CRUK,CRUK Commission
141,D,Discount
1815,DOT,DOTCOM POSTAGE
136537,DOT,NaN
2239,M,Manual
157195,PADS,PADS TO MATCH ALL CUSHIONS
45,POST,POSTAGE


We can appreciate that some descriptions are not related with the stock code, which indicates that there are some inconsistencies in the dataset. This could be due to data entry errors or other factors, and we will change this later in data preparation step. By addressing these inconsistencies, we can ensure that our analysis is based on accurate and reliable data, leading to more meaningful insights into customer segments and purchasing behavior.

In [242]:
quantity_negatives_special_stock_codes =   df[(df["StockCode"].isin(special_stock_codes)) & (df["Quantity"] < 0) & ((~df["Invoice"].str.startswith("C")))]["StockCode"].value_counts().head(10)

print(f"Number of negative quantities not related to invoice cancellations and special stock codes: {len(quantity_negatives_special_stock_codes)}")

Number of negative quantities not related to invoice cancellations and special stock codes: 0


There's no relation between the errors and these type of special stock codes, so we can conclude that the inconsistencies in the dataset are likely due to data entry errors or other factors, rather than being related to specific types of stock codes.

In [243]:
df.to_csv("../../data/processed/online_retail_II_without_duplicated.csv", index=False)

## Conclusion of the Data Quality Assessment

The dataset is overall usable for customer segmentation, but it contains several data quality issues that must be addressed before modeling. The most important findings are:

* **Missing Values**: Missing values are present, especially in Customer ID (24.93%), which is a critical field for segmentation. These rows should be excluded from customer-level analysis while preserving the transaction context when needed.
* **Description Cleanliness**: Description is missing in a small share of records (0.27%); these can be safely replaced with a placeholder such as "Unknown" or "No Description".
* **Duplicate Records**: Duplicate records were identified (10,149 rows, representing 1.87% of the dataset) and successfully removed using `drop_duplicates` to ensure transaction uniqueness and reduce noise without affecting the underlying business logic.
* **Temporal Reliability**: Invoice dates are valid and within the expected range (between 2009-12-01 and 2011-12-10), so no major date-related cleaning is required.
* **Cancellation Dynamics**: Negative quantities are mostly linked to invoice cancellations (Invoice numbers starting with "C"), accounting for 87.38% of negative quantities, which appear to be valid business events rather than errors.
* **Non-Cancellation Negatives**: A small portion of negative quantities (12.62%) does not correspond to cancellations and should be reviewed or removed as invalid transactions.
* **Price Anomalies**: Negative or zero prices are negligible (only 5 records, or 0.00093%) and can be treated as anomalies or removed safely to maintain calculation integrity.
* **Special Stock Codes**: Special stock codes such as POST, DOT, M, C2, D, S, CRUK, PADS, and B correspond to adjustments, discounts, postage, or other non-standard transactions. Some of these codes exhibited missing or misaligned descriptions that require normalization during data preparation.

In summary, the dataset is not dirty enough to invalidate the analysis, but it does require targeted cleaning and careful treatment of cancellations, non-standard stock codes, and missing customer information. Once these issues are addressed, the dataset will be suitable for a reliable customer segmentation workflow.

---

## Next Step: Data Preparation

The next phase, Data Preparation, will focus on transforming the dataset into a clean and analysis-ready structure for segmentation. The proposed path is:

* **Standardize columns and data types**
  * Convert dates to datetime format using the correct parsing layout.
  * Ensure numeric fields are numeric.
  * Clean column names for consistency.
* **Handle missing values**
  * Replace missing Description values with "Unknown".
  * Remove or flag rows with missing Customer ID for customer-level analysis.
* **Remove invalid or non-relevant transactions**
  * Drop duplicate rows based on unique transaction subsets.
  * Remove zero or negative price records.
  * Exclude non-cancellation negative quantity rows unless they are explicitly valid.
* **Create business flags**
  * Add `is_cancellation` flag based on Invoice prefix "C".
  * Add `special_stock_code` flag for adjustments, discounts, postage, and samples.
  * Add a `valid_transaction` flag for records retained in the analysis.
* **Normalize and enrich the dataset**
  * Standardize StockCode and Description values.
  * Handle special cases such as discounts and postage consistently.
  * Derive useful transaction-level features such as total sales, order value, and purchase recency indicators.
* **Build customer-level dataset**
  * Filter to valid customer transactions.
  * Aggregate by Customer ID.
  * Create segmentation features such as total spend, number of orders, average order value, recency, frequency, and product diversity.
* **Save the prepared dataset**
  * Export a cleaned transaction dataset.
  * Export a customer-level dataset ready for clustering and segmentation.

This preparation stage will ensure that the segmentation models are based on reliable, consistent, and business-meaningful data.